In [1]:
import pandas as pd
import numpy as np

In [ ]:
# 데이터 불러오기
df_to_train = pd.read_csv('train.csv')
df_to_test = pd.read_csv('test.csv')
submission = pd.read_csv('submission_sample.csv')
interp_df = pd.read_csv("nins_linera_interpolated_time.csv")

# 시간 바꿔주기
df_to_train['time'] = pd.to_datetime(df_to_train['time'])
df_to_test['time'] = pd.to_datetime(df_to_test['time'])
submission['time'] = pd.to_datetime(submission['time'])

df_to_train = df_to_train.drop(columns=['energy','type'], axis = 1)
df_to_test = df_to_test.drop(columns=['type'], axis = 1)

# 환경변수기반 클러스터 정보 불러오기
cluster_by_env = pd.read_csv('pv_cluster_map_by_input.csv')

# train, test의 발전소명 리스트로 보관
train_pv_list = df_to_train["pv_id"].unique().tolist()
test_pv_list = df_to_test["pv_id"].unique().tolist()

# cluster 값 넣기
df_to_train = df_to_train.merge(cluster_by_env[['pv_id', 'cluster']], on='pv_id', how='left')
df_to_test = df_to_test.merge(cluster_by_env[['pv_id', 'cluster']], on='pv_id', how='left')

### 전처리

In [3]:
env_vars = ["temp_a", "temp_b", "temp_max", "real_feel_temp", "real_feel_temp_shade",
    "uv_idx", "vis", "wind_spd_a", "wind_spd_b", "wind_gust_spd",
    "cloud_a", "cloud_b", "rel_hum", "humidity", "dew_point",
    "precip_1h", "rain", "snow", "wind_chill_temp", "ceiling",
    "pressure", "ground_press", "wind_dir_a", "wind_dir_b",
    "appr_temp", "temp_min"
]

In [4]:
# 상관계수 따른 변수 제거
def remove_feats(df, method = "remove"):
    if method == "remove":
        #cols_to_remove = ['appr_temp', 'real_feel_temp_shade', 'temp_max', 'temp_min', 'wind_chill_temp']                                   # 0.99
        #cols_to_remove = ['appr_temp', 'real_feel_temp_shade', 'temp_a', 'temp_b', 'temp_max', 'temp_min', 'wind_chill_temp']               # 0.97
        cols_not_to_remove = []   # 0.96
        cols_to_remove = [x for x in env_vars if x not in cols_not_to_remove]

        df_dropped = df.drop(columns = cols_to_remove)
    # elif method == "average":
        # 그룹화 해서 평균낸걸로 하나의 변수로 통합하기

    return df_dropped

# 시간 변환
def add_time_features(df):

    df_new = df.copy()

    # ===== ① year =====
    df_new['year'] = df_new['time'].dt.year
    
    # ===== ② day of year (월+일 통합 주기적 피처) =====
    dayofyear = df_new['time'].dt.dayofyear
    df_new['dayofyear_sin'] = np.sin(2 * np.pi * dayofyear / 365)
    df_new['dayofyear_cos'] = np.cos(2 * np.pi * dayofyear / 365)
    
    # ===== ③ time of day (일주기 피처) =====
    minutes_in_day = df_new['time'].dt.hour * 60 + df_new['time'].dt.minute
    df_new['time_sin'] = np.sin(2 * np.pi * minutes_in_day / (24 * 60))
    df_new['time_cos'] = np.cos(2 * np.pi * minutes_in_day / (24 * 60))
    

    def get_korean_quarter(dt):
        month = dt.month
        if month in [3, 4, 5]:
            return 1  # 봄
        elif month in [6, 7, 8]:
            return 2  # 여름
        elif month in [9, 10, 11]:
            return 3  # 가을
        else:
            return 4  # 겨울

    # 새로운 변수 추가
    df_new['quarter'] = df_new['time'].apply(get_korean_quarter)


    # df_new = df_new.drop('time', axis = 1)
    
    return df_new

# 일사량 보간 기간 제거
def remove_interpolated_segments(train_df, threshold=100):

    # 시간 형식 변환 (이미 되어있다면 건너뜀)
    train_df = train_df.copy()
    interp_df = pd.read_csv("nins_linera_interpolated_time.csv")
    interp_df['start_time'] = pd.to_datetime(interp_df['start_time'])
    interp_df['end_time'] = pd.to_datetime(interp_df['end_time'])
    
    # 삭제 마스크 생성
    mask = pd.Series(False, index=train_df.index)
    
    for pv, sub_df in interp_df.groupby('pv_id'):
        # threshold 이상 구간만 선택
        long_segments = sub_df[sub_df['length(m)'] >= threshold]
        if long_segments.empty:
            continue
        
        # 해당 발전소 데이터만 필터링
        pv_mask = train_df['pv_id'] == pv
        pv_times = train_df.loc[pv_mask, 'time']
        
        # 각 구간별로 삭제 구간 적용
        for _, row in long_segments.iterrows():
            mask |= pv_mask & (pv_times > row['start_time']) & (pv_times < row['end_time'])
    
    # 삭제된 데이터 제외
    result = train_df.loc[~mask].reset_index(drop=True)
    return result

# 좌표 및 클러스터별 통합
def average_nins_by_cluster(df, cluster_col='cluster', time_col='time', target_col='nins'):
    # coord1, coord2를 그룹화 기준에 추가
    group_keys = [cluster_col, 'coord1', 'coord2', time_col]

    # 군집+좌표+시간별 평균 계산
    avg_target = df.groupby(group_keys)[target_col].mean().reset_index()

    # 그룹화 키를 제외한 나머지 열만 가져오기
    other_cols = [c for c in df.columns if c not in [target_col, cluster_col, time_col, 'coord1', 'coord2']]
    first_obs = df.groupby(group_keys)[other_cols].first().reset_index()

    # 평균 타겟과 결합
    result = pd.merge(first_obs, avg_target, on=group_keys, how='left')

    return result


df_to_train     = add_time_features(df_to_train)
df_to_test      = add_time_features(df_to_test)
print("시간변수 추가 : 완료")

train_df_removed_interpol   = remove_interpolated_segments(df_to_train, threshold= 100)
train_clusterd              = average_nins_by_cluster(train_df_removed_interpol)
print("일사량 보간 시점 제거, 좌표 같은 발전소 평균 통합 : 완료")

train_clusterd_removed      = remove_feats(train_clusterd)
test_removed                = remove_feats(df_to_test)
print("환경 변수 제거 : 완료")

In [5]:
train_without_coords    = train_clusterd_removed.drop(columns = ['time',"uv_idx"])
test_without_coords     = test_removed.drop(columns = ['time',"uv_idx"])

In [6]:
train_without_coords.columns

Index(['cluster', 'coord1', 'coord2', 'pv_id', 'year', 'dayofyear_sin',
       'dayofyear_cos', 'time_sin', 'time_cos', 'quarter', 'nins'],
      dtype='object')

### 후처리

In [ ]:
# 밤시간 일사량 0으로
def zero_night_nins(df, start_hour=6, end_hour=18):
    # 시간(hour) 추출
    df = df.copy()
    df['hour'] = df['time'].dt.hour

    # 낮 시간 외는 nins = 0
    df.loc[~df['hour'].between(start_hour, end_hour), 'nins'] = 0

    # hour 컬럼 제거
    df.drop(columns=['hour'], inplace=True)
    
    return df

# 예측값 클립핑
# y_pred = y_pred.clip(lower=0, upper=1300)

## 학습

In [7]:
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error


# ---------- 설정 ----------
df = train_without_coords.copy()  # 학습용 데이터프레임
target_col = 'nins'               # 타깃 컬럼명
id_col = 'pv_id'                  # 발전소 식별자
val_ratio = 0.2                  # validation 비율

# ---------- 발전소 단위로 랜덤 분할 ----------
np.random.seed(47)
val_pv_ids = np.random.choice(df[id_col].unique(),
                              size=int(df[id_col].nunique() * val_ratio),
                              replace=False)

train_df = df[~df[id_col].isin(val_pv_ids)].copy()
val_df   = df[df[id_col].isin(val_pv_ids)].copy()

# ---------- 모델 입력 시 pv_id 제외 ----------
X_train = train_df.drop(columns=[target_col, id_col, 'cluster'])
y_train = train_df[target_col]

X_val = val_df.drop(columns=[target_col, id_col, 'cluster'])
y_val = val_df[target_col]

print(f"Train size: {len(train_df):,}, Validation size: {len(val_df):,}")

Train size: 7,358,774, Validation size: 1,885,067


In [ ]:
# LightGBM Dataset 구성
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)


# =========================================================
# Optuna 목적 함수 정의
# =========================================================
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'device': 'gpu',
        'feature_pre_filter': False,  # 🔹 반드시 추가
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 4, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 500),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'verbosity': -1,
    }

    model = lgb.train(
        params=params,
        train_set=train_data,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        num_boost_round=2000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=300),
            lgb.log_evaluation(period=0)
        ]
    )

    preds = model.predict(X_val, num_iteration=model.best_iteration)
    score = mean_absolute_error(y_val, preds)
    return score


# =========================================================
# Optuna 실행
# =========================================================
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)


# =========================================================
# 최적 파라미터로 최종 학습
# =========================================================
best_params = study.best_params
best_params.update({
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbosity': -1,
})

[I 2025-11-09 21:08:39,455] A new study created in memory with name: no-name-2fa38518-62ad-4398-afda-d135d7d05a72


  0%|          | 0/10 [00:00<?, ?it/s]

Training until validation scores don't improve for 300 rounds
Did not meet early stopping. Best iteration is:
[2000]	train's l1: 42.1247	valid's l1: 52.5787
[I 2025-11-09 21:13:10,332] Trial 0 finished with value: 52.5786856133974 and parameters: {'learning_rate': 0.020721519914566427, 'num_leaves': 71, 'max_depth': 15, 'min_data_in_leaf': 283, 'feature_fraction': 0.9994984654764805, 'bagging_fraction': 0.764323639550918, 'bagging_freq': 1, 'lambda_l1': 0.5062496758115769, 'lambda_l2': 0.10126463946551871}. Best is trial 0 with value: 52.5786856133974.
Training until validation scores don't improve for 300 rounds
Did not meet early stopping. Best iteration is:
[2000]	train's l1: 43.3298	valid's l1: 55.4853
[I 2025-11-09 21:18:15,984] Trial 1 finished with value: 55.48530235802422 and parameters: {'learning_rate': 0.036243639078868906, 'num_leaves': 111, 'max_depth': 15, 'min_data_in_leaf': 157, 'feature_fraction': 0.7670486814818424, 'bagging_fraction': 0.8143958388132997, 'bagging_fre

In [ ]:
final_model = lgb.train(
    params=best_params,
    train_set=train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
)

# 1️⃣ X에서 결측치 제거
X_test   = test_without_coords.drop(columns=[target_col, 'cluster'])
test_index = X_test.index  # 결측치 제거 후 남은 index 저장

# 2️⃣ 모델 예측
y_pred = final_model.predict(X_test, num_iteration=final_model.best_iteration)


# 4️⃣ index 기준으로 nins 값 업데이트
submission.loc[test_index, 'nins'] = y_pred

# 5️⃣ 확인
print(submission.head(50))